# Module 03: Power Automate Expression Reference

## Learning Objectives

By completing this notebook, you will:
1. Understand Power Automate's expression language by mapping each function to its Python equivalent
2. Build a Python evaluator that parses and executes Power Automate expression strings
3. Use a side-by-side comparison table to select the right function for any data task
4. Validate Power Automate expression strings against known-correct patterns

## Prerequisites

- Python 3.9+
- Module 03 Guide 01 (Dynamic Content and Expressions) completed
- Familiarity with Python strings, lists, and datetime objects

## Estimated Time: 15 minutes

---

**Why Python for Power Automate concepts?**

Power Automate expressions are a no-code formula language. Python is the closest general-purpose language to that formula style — it treats functions as first-class values and has rich built-in string and datetime support. Seeing the two side by side makes the Power Automate expression language concrete and builds intuition you can apply directly when writing expressions in the flow designer.

In [ ]:
# Setup: standard library only — no external dependencies required
import re
import json
import datetime
from typing import Any, Callable

print("Setup complete. Python", __import__('sys').version.split()[0])

## 1. String Functions

Power Automate string functions map directly to Python string methods, with one key difference:
Power Automate uses **function call syntax** for everything — there are no method calls.

In Python you write `'hello'.upper()`. In Power Automate you write `toUpper('hello')`.
Both produce `'HELLO'`.

In [ ]:
# Power Automate string functions implemented in Python
# Each function signature matches the Power Automate argument order exactly.

def pa_concat(*args: str) -> str:
    """concat(text1, text2, ...) — join any number of strings.
    
    Power Automate: concat('Hello, ', firstName, '!')
    Python equiv:  ''.join(['Hello, ', first_name, '!'])
    """
    return ''.join(str(a) for a in args)


def pa_substring(text: str, start_index: int, length: int) -> str:
    """substring(text, startIndex, length) — extract a portion of a string.
    
    Power Automate: substring('Invoice-2024', 8, 4)  →  '2024'
    Python equiv:  'Invoice-2024'[8:8+4]
    Note: startIndex is zero-based, exactly like Python slicing.
    """
    return text[start_index : start_index + length]


def pa_replace(text: str, find: str, replacement: str) -> str:
    """replace(text, oldValue, newValue) — replace all occurrences.
    
    Power Automate: replace('order_status', '_', ' ')  →  'order status'
    Python equiv:  'order_status'.replace('_', ' ')
    """
    return text.replace(find, replacement)


def pa_split(text: str, delimiter: str) -> list:
    """split(text, delimiter) — split a string into an array.
    
    Power Automate: split('a,b,c', ',')  →  ['a', 'b', 'c']
    Python equiv:  'a,b,c'.split(',')
    """
    return text.split(delimiter)


def pa_to_lower(text: str) -> str:
    """toLower(text) — convert to lowercase.
    
    Power Automate: toLower('HELLO')  →  'hello'
    Python equiv:  'HELLO'.lower()
    """
    return text.lower()


def pa_to_upper(text: str) -> str:
    """toUpper(text) — convert to uppercase.
    
    Power Automate: toUpper('hello')  →  'HELLO'
    Python equiv:  'hello'.upper()
    """
    return text.upper()


def pa_trim(text: str) -> str:
    """trim(text) — remove leading and trailing whitespace.
    
    Power Automate: trim('  hello  ')  →  'hello'
    Python equiv:  '  hello  '.strip()
    Note: Power Automate trim() strips both ends. Python also has lstrip()/rstrip()
    for one-sided trimming, but Power Automate has no equivalent — use substring() instead.
    """
    return text.strip()


# Run all string functions and verify expected outputs
assert pa_concat('Hello, ', 'Priya', '!') == 'Hello, Priya!'
assert pa_substring('Invoice-2024-001', 8, 4) == '2024'
assert pa_replace('order_status_report', '_', ' ') == 'order status report'
assert pa_split('apple,banana,cherry', ',') == ['apple', 'banana', 'cherry']
assert pa_to_lower('HELLO WORLD') == 'hello world'
assert pa_to_upper('hello world') == 'HELLO WORLD'
assert pa_trim('  hello  ') == 'hello'

print('All string function tests passed.')

# Demonstrate the most common real-world pattern:
# clean a user-entered name before storing it
raw_name = '  PRIYA SHARMA  '
cleaned = pa_trim(pa_to_lower(raw_name))
print(f'Raw: {repr(raw_name)}')
print(f'Cleaned: {repr(cleaned)}')
print()
print('Power Automate equivalent: trim(toLower(triggerBody()[\'FullName\']))')

## 2. Date and Time Functions

Power Automate stores datetimes as ISO 8601 strings internally
(`'2024-11-15T09:32:00.0000000Z'`). Functions like `formatDateTime()` parse
this string, apply the format, and return a new string.

Python's `datetime` module follows the same model. The main difference is
format codes: Power Automate uses .NET codes (`yyyy`, `MM`, `dd`, `HH`),
while Python uses strftime codes (`%Y`, `%m`, `%d`, `%H`).

In [ ]:
# Format code mapping: Power Automate .NET codes → Python strftime codes
# This mapping is used inside our pa_format_date_time() implementation.

DOTNET_TO_STRFTIME: dict[str, str] = {
    # Year
    'yyyy': '%Y',   # 2024
    'yy':   '%y',   # 24
    # Month
    'MMMM': '%B',   # November
    'MMM':  '%b',   # Nov
    'MM':   '%m',   # 11
    'M':    '%-m',  # 11 (no leading zero — Linux only; use %m on Windows)
    # Day
    'dddd': '%A',   # Friday
    'ddd':  '%a',   # Fri
    'dd':   '%d',   # 15
    'd':    '%-d',  # 15 (no leading zero)
    # Hour
    'HH':   '%H',   # 09 (24-hour)
    'H':    '%-H',  # 9  (24-hour, no leading zero)
    'hh':   '%I',   # 09 (12-hour)
    'h':    '%-I',  # 9  (12-hour, no leading zero)
    # Minute / Second
    'mm':   '%M',   # 32
    'ss':   '%S',   # 00
    # AM/PM
    'tt':   '%p',   # AM / PM
}


def _dotnet_to_py_format(dotnet_fmt: str) -> str:
    """Convert a .NET custom datetime format string to a Python strftime format."""
    # Sort by length descending to avoid partial replacements (e.g. 'MM' before 'M')
    py_fmt = dotnet_fmt
    for dotnet_code, py_code in sorted(DOTNET_TO_STRFTIME.items(), key=lambda x: -len(x[0])):
        py_fmt = py_fmt.replace(dotnet_code, py_code)
    return py_fmt


def pa_utc_now(format_str: str | None = None) -> str:
    """utcNow(format?) — return the current UTC datetime.
    
    Power Automate: utcNow()                   →  '2024-11-15T09:32:00.0000000Z'
    Power Automate: utcNow('yyyy-MM-dd')       →  '2024-11-15'
    Python equiv:   datetime.utcnow().isoformat() + 'Z'
    """
    now = datetime.datetime.utcnow()
    if format_str:
        return now.strftime(_dotnet_to_py_format(format_str))
    return now.strftime('%Y-%m-%dT%H:%M:%S.0000000Z')


def pa_format_date_time(timestamp: str, format_str: str) -> str:
    """formatDateTime(timestamp, format) — format a datetime string.
    
    Power Automate: formatDateTime('2024-11-15T09:32:00Z', 'MMMM d, yyyy')
                    →  'November 15, 2024'
    Python equiv:   datetime.fromisoformat(...).strftime('%B %-d, %Y')
    """
    # Parse ISO 8601 — strip trailing Z and fractional seconds for compatibility
    clean = timestamp.rstrip('Z').split('.')[0]
    dt = datetime.datetime.fromisoformat(clean)
    py_fmt = _dotnet_to_py_format(format_str)
    return dt.strftime(py_fmt)


def pa_add_days(timestamp: str, days: int, format_str: str | None = None) -> str:
    """addDays(timestamp, days, format?) — add or subtract days.
    
    Power Automate: addDays(utcNow(), -7, 'yyyy-MM-dd')  →  '2024-11-08'
    Python equiv:   (datetime.utcnow() + timedelta(days=-7)).strftime('%Y-%m-%d')
    """
    clean = timestamp.rstrip('Z').split('.')[0]
    dt = datetime.datetime.fromisoformat(clean)
    result = dt + datetime.timedelta(days=days)
    if format_str:
        return result.strftime(_dotnet_to_py_format(format_str))
    return result.strftime('%Y-%m-%dT%H:%M:%S.0000000Z')


# Test datetime functions with a fixed timestamp for determinism
TEST_TIMESTAMP = '2024-11-15T09:32:00Z'

assert pa_format_date_time(TEST_TIMESTAMP, 'yyyy-MM-dd') == '2024-11-15'
assert pa_format_date_time(TEST_TIMESTAMP, 'MMMM d, yyyy') == 'November 15, 2024'
assert pa_add_days(TEST_TIMESTAMP, 30, 'yyyy-MM-dd') == '2024-12-15'
assert pa_add_days(TEST_TIMESTAMP, -7, 'yyyy-MM-dd') == '2024-11-08'

print('All datetime function tests passed.')
print()
print('formatDateTime examples:')
for fmt in ['yyyy-MM-dd', 'MMMM d, yyyy', 'dddd, MMMM d', 'MM/dd/yyyy', 'HH:mm']:
    print(f'  formatDateTime(timestamp, \'{fmt}\')  →  {pa_format_date_time(TEST_TIMESTAMP, fmt)!r}')

## 3. Logical Functions

Power Automate's logical functions are function-based wrappers around
standard comparison operators. Python's operators (`==`, `>`, `and`, `or`, `not`)
are more concise, but the Power Automate functions are necessary because
expressions are strings — they cannot use Python's infix operator syntax.

In [ ]:
# Logical and comparison functions

def pa_if(condition: bool, value_if_true: Any, value_if_false: Any) -> Any:
    """if(condition, valueIfTrue, valueIfFalse) — ternary selection.
    
    Power Automate: if(greater(amount, 10000), 'VP required', 'Auto-approved')
    Python equiv:   'VP required' if amount > 10000 else 'Auto-approved'
    
    IMPORTANT: if() is for VALUE SELECTION only.
    To run DIFFERENT ACTIONS based on a condition, use the Condition control
    shape in the flow designer — not this function.
    """
    return value_if_true if condition else value_if_false


def pa_equals(value1: Any, value2: Any) -> bool:
    """equals(value1, value2) — strict equality check.
    
    Power Automate: equals(status, 'Approved')
    Python equiv:   status == 'Approved'
    """
    return value1 == value2


def pa_greater(value1: Any, value2: Any) -> bool:
    """greater(value1, value2) — value1 > value2.
    
    Power Automate: greater(amount, 5000)
    Python equiv:   amount > 5000
    """
    return value1 > value2


def pa_less(value1: Any, value2: Any) -> bool:
    """less(value1, value2) — value1 < value2.
    
    Power Automate: less(hour, 12)
    Python equiv:   hour < 12
    """
    return value1 < value2


def pa_and(condition1: bool, condition2: bool) -> bool:
    """and(cond1, cond2) — both conditions must be true.
    
    Power Automate: and(equals(status, 'Open'), greater(daysOpen, 14))
    Python equiv:   status == 'Open' and days_open > 14
    Note: and() takes exactly TWO arguments. For three conditions, nest:
    and(cond1, and(cond2, cond3))
    """
    return condition1 and condition2


def pa_or(condition1: bool, condition2: bool) -> bool:
    """or(cond1, cond2) — at least one condition must be true.
    
    Power Automate: or(equals(priority, 'High'), equals(priority, 'Critical'))
    Python equiv:   priority in ('High', 'Critical')
    """
    return condition1 or condition2


def pa_not(condition: bool) -> bool:
    """not(condition) — invert a boolean.
    
    Power Automate: not(empty(attachments))
    Python equiv:   not not attachments  (or: bool(attachments))
    """
    return not condition


def pa_coalesce(*args: Any) -> Any:
    """coalesce(val1, val2, ...) — return the first non-null value.
    
    Power Automate: coalesce(managerEmail, defaultApproverEmail)
    Python equiv:   manager_email or default_approver_email
    Note: None and empty string ('') are both treated as null by coalesce() in Power Automate.
    This implementation matches that behaviour.
    """
    for arg in args:
        if arg is not None and arg != '':
            return arg
    return None


# Tests
assert pa_if(pa_greater(15000, 10000), 'VP required', 'Auto-approved') == 'VP required'
assert pa_if(pa_greater(5000, 10000), 'VP required', 'Auto-approved') == 'Auto-approved'
assert pa_equals('Approved', 'Approved') is True
assert pa_equals('Approved', 'Pending') is False
assert pa_and(True, True) is True
assert pa_and(True, False) is False
assert pa_or(False, True) is True
assert pa_not(True) is False
assert pa_coalesce(None, '', 'default@contoso.com') == 'default@contoso.com'
assert pa_coalesce('alice@contoso.com', 'default@contoso.com') == 'alice@contoso.com'

print('All logical function tests passed.')

# Real-world example: time-of-day greeting
hour = int(pa_format_date_time(TEST_TIMESTAMP, 'H'))  # 9
greeting = pa_if(pa_less(hour, 12), 'Good morning', 'Good afternoon')
print(f'\nHour: {hour} → Greeting: {greeting!r}')
print('Power Automate: if(less(int(formatDateTime(utcNow(), \'H\')), 12), \'Good morning\', \'Good afternoon\')')

## 4. Type Conversion Functions

External systems frequently send numbers as strings, or booleans as the
string `"true"`. Power Automate is strictly typed at runtime — arithmetic
on a string fails. Type conversion functions solve this.

Python's built-in `int()`, `float()`, `str()`, `bool()`, and `json.loads()`
are direct equivalents.

In [ ]:
# Type conversion functions

def pa_int(value: Any) -> int:
    """int(value) — convert to integer.
    
    Power Automate: int(triggerBody()?['Quantity'])  converts '5' → 5
    Python equiv:   int(value)
    Common use: SharePoint number columns that arrive as strings.
    """
    return int(value)


def pa_float(value: Any) -> float:
    """float(value) — convert to floating-point number.
    
    Power Automate: float(triggerBody()?['UnitPrice'])  converts '9.99' → 9.99
    Python equiv:   float(value)
    """
    return float(value)


def pa_string(value: Any) -> str:
    """string(value) — convert to string.
    
    Power Automate: string(variables('InvoiceNumber'))  converts 1042 → '1042'
    Python equiv:   str(value)
    Required before using a number inside concat() or other string functions.
    """
    return str(value)


def pa_bool(value: Any) -> bool:
    """bool(value) — convert string 'true'/'false' to boolean.
    
    Power Automate: bool(triggerBody()?['IsUrgent'])  converts 'true' → true
    Python equiv:   value.lower() == 'true' if isinstance(value, str) else bool(value)
    """
    if isinstance(value, str):
        return value.strip().lower() == 'true'
    return bool(value)


def pa_json(value: str) -> Any:
    """json(value) — parse a JSON string into an object.
    
    Power Automate: json(body('HTTP_Request'))  →  accessible object
    Python equiv:   json.loads(value)
    After this, fields are accessible with ?['key'] notation in Power Automate,
    or standard dict access in Python.
    """
    return json.loads(value)


# Tests
assert pa_int('5') == 5
assert pa_int(5.9) == 5       # truncates, does not round
assert pa_float('9.99') == 9.99
assert pa_string(1042) == '1042'
assert pa_string(3.14) == '3.14'
assert pa_bool('true') is True
assert pa_bool('false') is False
assert pa_bool('True') is True  # case-insensitive
assert pa_json('{"invoiceId": "INV-001", "amount": 1500}')['amount'] == 1500

print('All type conversion function tests passed.')

# Real-world example: compute invoice total from string inputs
quantity_str = '12'    # came from a web form
unit_price_str = '24.50'  # came from a CSV column

# In Power Automate: mul(int(quantity), float(unitPrice))
total = pa_int(quantity_str) * pa_float(unit_price_str)
print(f'\nInvoice total: {quantity_str} × {unit_price_str} = {total}')
print(f'Formatted:     {pa_string(total)}')
print()
print('Power Automate: string(mul(int(triggerBody()[\'Quantity\']), float(triggerBody()[\'UnitPrice\'])))')

## 5. Collection Functions

Collection functions operate on arrays (and strings, which Power Automate
treats as sequences of characters for length/contains purposes).
They map to Python built-ins and list operations.

In [ ]:
# Collection functions

def pa_length(collection: list | str) -> int:
    """length(collection) — number of items in an array, or characters in a string.
    
    Power Automate: length(variables('ApproverList'))  →  3
    Power Automate: length(triggerBody()?['Subject'])  →  character count
    Python equiv:   len(collection)
    """
    return len(collection)


def pa_first(collection: list) -> Any:
    """first(collection) — return the first item.
    
    Power Automate: first(variables('ApproverList'))  →  'alice@contoso.com'
    Python equiv:   collection[0]
    Raises IndexError if the collection is empty — check with empty() first.
    """
    return collection[0]


def pa_last(collection: list) -> Any:
    """last(collection) — return the last item.
    
    Power Automate: last(variables('ApproverList'))  →  'carol@contoso.com'
    Python equiv:   collection[-1]
    """
    return collection[-1]


def pa_contains(collection: list | str, value: Any) -> bool:
    """contains(collection, value) — check membership.
    
    Power Automate: contains(approvedStatuses, status)  →  true/false
    Power Automate: contains(subject, 'URGENT')         →  true/false
    Python equiv:   value in collection
    Works on both arrays (checks for item) and strings (checks for substring).
    """
    return value in collection


def pa_empty(collection: list | str | None) -> bool:
    """empty(collection) — true if the collection has no items.
    
    Power Automate: empty(triggerBody()?['Attachments'])  →  true/false
    Python equiv:   not collection  (or: len(collection) == 0)
    Also returns true for None.
    """
    if collection is None:
        return True
    return len(collection) == 0


# Tests
approvers = ['alice@contoso.com', 'bob@contoso.com', 'carol@contoso.com']

assert pa_length(approvers) == 3
assert pa_length('Hello') == 5
assert pa_first(approvers) == 'alice@contoso.com'
assert pa_last(approvers) == 'carol@contoso.com'
assert pa_contains(approvers, 'bob@contoso.com') is True
assert pa_contains(approvers, 'dave@contoso.com') is False
assert pa_contains('URGENT: system down', 'URGENT') is True
assert pa_empty([]) is True
assert pa_empty(approvers) is False
assert pa_empty(None) is True

print('All collection function tests passed.')

# Real-world pattern: guard clause before using first()
attachments = []  # simulates no attachments

if pa_not(pa_empty(attachments)):
    first_attachment = pa_first(attachments)
    print(f'\nFirst attachment: {first_attachment}')
else:
    print('\nNo attachments — skipping attachment processing')

print('Power Automate: if(not(empty(triggerBody()[\'Attachments\'])), first(triggerBody()[\'Attachments\']), null)')

## 6. Side-by-Side Comparison Table

This table is your quick reference when writing expressions in Power Automate.
Find the Python operation you know and read across to the Power Automate equivalent.

In [ ]:
# Generate a formatted comparison table

COMPARISON_TABLE: list[dict] = [
    # --- String ---
    {
        'category': 'String',
        'python': "'Hello, ' + name + '!'",
        'power_automate': "concat('Hello, ', name, '!')",
        'notes': 'PA has no + operator; always use concat()'
    },
    {
        'category': 'String',
        'python': "text[8:12]",
        'power_automate': "substring(text, 8, 4)",
        'notes': 'PA uses (start, length); Python uses (start, end)'
    },
    {
        'category': 'String',
        'python': "text.replace('_', ' ')",
        'power_automate': "replace(text, '_', ' ')",
        'notes': 'Same semantics, function vs method'
    },
    {
        'category': 'String',
        'python': "text.split(',')",
        'power_automate': "split(text, ',')",
        'notes': 'Returns array; feed into Apply to each'
    },
    {
        'category': 'String',
        'python': "text.lower()",
        'power_automate': "toLower(text)",
        'notes': 'Normalise before comparison'
    },
    {
        'category': 'String',
        'python': "text.upper()",
        'power_automate': "toUpper(text)",
        'notes': ''
    },
    {
        'category': 'String',
        'python': "text.strip()",
        'power_automate': "trim(text)",
        'notes': 'Both ends; no lstrip/rstrip equivalent'
    },
    # --- Date / Time ---
    {
        'category': 'Date/Time',
        'python': "datetime.utcnow().isoformat() + 'Z'",
        'power_automate': "utcNow()",
        'notes': 'Always UTC in PA'
    },
    {
        'category': 'Date/Time',
        'python': "dt.strftime('%B %-d, %Y')",
        'power_automate': "formatDateTime(ts, 'MMMM d, yyyy')",
        'notes': '.NET format codes not strftime codes'
    },
    {
        'category': 'Date/Time',
        'python': "dt + timedelta(days=30)",
        'power_automate': "addDays(ts, 30)",
        'notes': 'Negative days go backward'
    },
    # --- Logical ---
    {
        'category': 'Logical',
        'python': "'Yes' if condition else 'No'",
        'power_automate': "if(condition, 'Yes', 'No')",
        'notes': 'Value selection only, not flow branching'
    },
    {
        'category': 'Logical',
        'python': "a == b",
        'power_automate': "equals(a, b)",
        'notes': ''
    },
    {
        'category': 'Logical',
        'python': "cond1 and cond2",
        'power_automate': "and(cond1, cond2)",
        'notes': 'Exactly 2 args; nest for 3+'
    },
    {
        'category': 'Logical',
        'python': "val1 or val2 or default",
        'power_automate': "coalesce(val1, val2, default)",
        'notes': 'Returns first non-null'
    },
    # --- Type conversion ---
    {
        'category': 'Type',
        'python': "int(value)",
        'power_automate': "int(value)",
        'notes': 'Same name, same behaviour'
    },
    {
        'category': 'Type',
        'python': "float(value)",
        'power_automate': "float(value)",
        'notes': 'Same name, same behaviour'
    },
    {
        'category': 'Type',
        'python': "str(value)",
        'power_automate': "string(value)",
        'notes': "PA uses 'string', not 'str'"
    },
    {
        'category': 'Type',
        'python': "json.loads(value)",
        'power_automate': "json(value)",
        'notes': 'Unlocks field tokens in dynamic content panel'
    },
    # --- Collection ---
    {
        'category': 'Collection',
        'python': "len(collection)",
        'power_automate': "length(collection)",
        'notes': 'Works on arrays and strings'
    },
    {
        'category': 'Collection',
        'python': "collection[0]",
        'power_automate': "first(collection)",
        'notes': 'Check empty() first'
    },
    {
        'category': 'Collection',
        'python': "collection[-1]",
        'power_automate': "last(collection)",
        'notes': ''
    },
    {
        'category': 'Collection',
        'python': "value in collection",
        'power_automate': "contains(collection, value)",
        'notes': 'Works on arrays and strings'
    },
    {
        'category': 'Collection',
        'python': "not collection",
        'power_automate': "empty(collection)",
        'notes': 'Also true for None'
    },
]

# Print formatted table
header = f"{'Category':<12} {'Python':<40} {'Power Automate':<45} {'Notes'}"
print(header)
print('-' * len(header))

current_category = None
for row in COMPARISON_TABLE:
    cat = row['category'] if row['category'] != current_category else ''
    current_category = row['category']
    print(f"{cat:<12} {row['python']:<40} {row['power_automate']:<45} {row['notes']}")

print(f'\nTotal: {len(COMPARISON_TABLE)} function mappings across 5 categories')

## 7. Power Automate Expression Evaluator

The following class provides a simple evaluator that parses Power Automate
expression syntax and executes it using the Python implementations above.

This is an educational tool — it handles a representative subset of the
expression language to make the evaluation model concrete. It is not a
full implementation of Workflow Definition Language.

In [ ]:
class PowerAutomateEvaluator:
    """Evaluates a subset of Power Automate expression strings.
    
    Supports: string literals (single quotes), integer and float literals,
    boolean literals, nested function calls for all 22 functions implemented
    in this notebook.
    
    Not supported (intentionally): dynamic content references (triggerBody(),
    variables(), outputs()), XML functions, network calls.
    """

    def __init__(self, context: dict[str, Any] | None = None):
        """Initialise with an optional context dict for variable lookup.
        
        Parameters
        ----------
        context : dict, optional
            Maps variable names to values. Used when the expression
            references 'variables("name")' or field names.
        """
        self.context = context or {}

        # Registry: lowercase function name → callable
        self._functions: dict[str, Callable] = {
            'concat':         pa_concat,
            'substring':      pa_substring,
            'replace':        pa_replace,
            'split':          pa_split,
            'tolower':        pa_to_lower,
            'toupper':        pa_to_upper,
            'trim':           pa_trim,
            'utcnow':         pa_utc_now,
            'formatdatetime': pa_format_date_time,
            'adddays':        pa_add_days,
            'if':             pa_if,
            'equals':         pa_equals,
            'greater':        pa_greater,
            'less':           pa_less,
            'and':            pa_and,
            'or':             pa_or,
            'not':            pa_not,
            'coalesce':       pa_coalesce,
            'int':            pa_int,
            'float':          pa_float,
            'string':         pa_string,
            'bool':           pa_bool,
            'json':           pa_json,
            'length':         pa_length,
            'first':          pa_first,
            'last':           pa_last,
            'contains':       pa_contains,
            'empty':          pa_empty,
        }

    def evaluate(self, expression: str) -> Any:
        """Parse and evaluate a Power Automate expression string.
        
        Parameters
        ----------
        expression : str
            A Power Automate expression, e.g. "toLower(concat('Hello', name))"
        
        Returns
        -------
        Any
            The result of evaluating the expression.
        """
        return self._parse(expression.strip())

    def _parse(self, expr: str) -> Any:
        """Recursively parse an expression string."""
        expr = expr.strip()

        # String literal: 'text'
        if expr.startswith("'") and expr.endswith("'"):
            return expr[1:-1]

        # Boolean literals
        if expr == 'true':
            return True
        if expr == 'false':
            return False
        if expr == 'null':
            return None

        # Numeric literals
        try:
            if '.' in expr:
                return float(expr)
            return int(expr)
        except ValueError:
            pass

        # Function call: name(...)
        func_match = re.match(r'^([a-zA-Z][a-zA-Z0-9]*)\((.*)\)$', expr, re.DOTALL)
        if func_match:
            func_name = func_match.group(1).lower()
            args_str = func_match.group(2).strip()

            if func_name not in self._functions:
                raise ValueError(
                    f"Unknown function: '{func_match.group(1)}'. "
                    f"Supported: {sorted(self._functions.keys())}"
                )

            # Parse comma-separated arguments, respecting nested parentheses and quoted strings
            args = self._split_args(args_str)
            evaluated_args = [self._parse(a.strip()) for a in args]
            return self._functions[func_name](*evaluated_args)

        # Context lookup: treat as a variable name
        if expr in self.context:
            return self.context[expr]

        raise ValueError(
            f"Cannot parse expression fragment: {expr!r}. "
            f"Check for mismatched quotes or parentheses."
        )

    def _split_args(self, args_str: str) -> list[str]:
        """Split a comma-separated argument string, respecting nesting."""
        args: list[str] = []
        depth = 0
        in_quote = False
        current: list[str] = []

        for char in args_str:
            if char == "'" and depth == 0:
                in_quote = not in_quote
                current.append(char)
            elif char == '(' and not in_quote:
                depth += 1
                current.append(char)
            elif char == ')' and not in_quote:
                depth -= 1
                current.append(char)
            elif char == ',' and depth == 0 and not in_quote:
                args.append(''.join(current).strip())
                current = []
            else:
                current.append(char)

        if current:
            args.append(''.join(current).strip())

        return [a for a in args if a]  # remove empty strings from trailing commas


# Demonstration: evaluate real Power Automate expressions
ev = PowerAutomateEvaluator(context={'firstName': 'Priya', 'status': 'Approved'})

test_expressions = [
    ("concat('Hello, ', 'world')",                            'Hello, world'),
    ("toLower('HELLO')",                                      'hello'),
    ("trim('  hello  ')",                                     'hello'),
    ("substring('Invoice-2024-001', 8, 4)",                   '2024'),
    ("replace('a_b_c', '_', '-')",                             'a-b-c'),
    ("int('42')",                                             42),
    ("string(100)",                                           '100'),
    ("bool('true')",                                          True),
    ("equals('Approved', 'Approved')",                        True),
    ("if(equals('Approved', 'Approved'), 'Yes', 'No')",       'Yes'),
    ("if(greater(5000, 10000), 'VP required', 'Auto-ok')",    'Auto-ok'),
    ("coalesce('', null, 'default')",                         None),  # '' treated as null
    ("length('hello')",                                       5),
    ("contains('URGENT: down', 'URGENT')",                    True),
    ("empty('')",                                             True),
    ("not(empty('hello'))",                                   True),
    ("trim(toLower('  PRIYA  '))",                            'priya'),
    ("formatDateTime('2024-11-15T09:32:00Z', 'yyyy-MM-dd')",  '2024-11-15'),
    ("addDays('2024-11-15T00:00:00Z', 30, 'yyyy-MM-dd')",     '2024-12-15'),
]

print('Expression Evaluator Results:')
print('=' * 80)
all_passed = True
for expr, expected in test_expressions:
    result = ev.evaluate(expr)
    status = 'PASS' if result == expected else 'FAIL'
    if status == 'FAIL':
        all_passed = False
    short_expr = (expr[:55] + '...') if len(expr) > 58 else expr
    print(f'  [{status}] {short_expr:<58} → {result!r}')

print()
print('All tests passed.' if all_passed else 'Some tests failed — review output above.')

## 8. Expression Validator

The following validator checks whether a Power Automate expression string
follows the syntactic rules that most commonly trip up beginners:

1. String literals use single quotes, not double quotes
2. Parentheses are balanced
3. The outermost function name is a known Power Automate function
4. No `@` prefix (that prefix is for Filter array advanced mode, not the Expression editor)

In [ ]:
KNOWN_FUNCTIONS: frozenset[str] = frozenset([
    'concat', 'substring', 'replace', 'split', 'toLower', 'toUpper', 'trim',
    'startsWith', 'endsWith', 'indexOf', 'lastIndexOf', 'nthIndexOf',
    'utcNow', 'formatDateTime', 'addDays', 'addHours', 'addMinutes',
    'addSeconds', 'addMonths', 'addYears', 'convertTimeZone',
    'if', 'equals', 'greater', 'greaterOrEquals', 'less', 'lessOrEquals',
    'and', 'or', 'not', 'coalesce',
    'int', 'float', 'string', 'bool', 'json', 'xml', 'base64', 'base64ToString',
    'length', 'first', 'last', 'contains', 'empty', 'union', 'intersection',
    'skip', 'take', 'reverse', 'range', 'chunk',
    'min', 'max', 'add', 'sub', 'mul', 'div', 'mod', 'rand', 'abs', 'power',
    'outputs', 'triggerBody', 'triggerOutputs', 'variables', 'items',
    'body', 'actions', 'item', 'workflow', 'guid', 'uriComponent',
])


class ExpressionValidationResult:
    """Result of validating a Power Automate expression."""

    def __init__(self, expression: str):
        self.expression = expression
        self.errors: list[str] = []
        self.warnings: list[str] = []

    @property
    def is_valid(self) -> bool:
        return len(self.errors) == 0

    def __repr__(self) -> str:
        status = 'VALID' if self.is_valid else 'INVALID'
        parts = [f'[{status}] {self.expression!r}']
        for err in self.errors:
            parts.append(f'  ERROR: {err}')
        for warn in self.warnings:
            parts.append(f'  WARNING: {warn}')
        return '\n'.join(parts)


def validate_expression(expression: str) -> ExpressionValidationResult:
    """Validate a Power Automate expression string for common errors.
    
    Checks performed:
    1. No double-quoted string literals (must use single quotes)
    2. Balanced parentheses
    3. No leading @ symbol (that is Filter array advanced mode syntax)
    4. Outer function name is in the known-functions list
    5. No empty function argument lists where arguments are required
    
    Parameters
    ----------
    expression : str
        The expression string to validate.
    
    Returns
    -------
    ExpressionValidationResult
    """
    result = ExpressionValidationResult(expression)
    expr = expression.strip()

    # Check 1: leading @ prefix
    if expr.startswith('@'):
        result.errors.append(
            'Remove the leading @. The @ prefix is for Filter array advanced mode. '
            'In the Expression editor tab, omit the @ entirely.'
        )
        expr = expr.lstrip('@')  # continue checking the rest

    # Check 2: double-quoted strings (look for content outside single-quoted regions)
    # Remove single-quoted regions first, then check for " remaining
    outside_quotes = re.sub(r"'[^']*'", "''", expr)
    if '"' in outside_quotes:
        result.errors.append(
            'Found double quotes ("). Power Automate expressions require single quotes for '
            "string literals. Replace \"hello\" with 'hello'."
        )

    # Check 3: balanced parentheses
    depth = 0
    in_quote = False
    for i, ch in enumerate(expr):
        if ch == "'":
            in_quote = not in_quote
        elif not in_quote:
            if ch == '(':
                depth += 1
            elif ch == ')':
                depth -= 1
                if depth < 0:
                    result.errors.append(
                        f'Unexpected closing parenthesis at position {i}. '
                        'Count your opening and closing parentheses — they must match.'
                    )
                    depth = 0  # reset to continue checking

    if depth > 0:
        result.errors.append(
            f'{depth} unclosed parenthesis{"" if depth == 1 else "es"} detected. '
            'Every opening ( needs a matching closing ).'
        )

    # Check 4: outer function name known
    func_match = re.match(r'^([a-zA-Z][a-zA-Z0-9]*)\(', expr)
    if func_match:
        outer_func = func_match.group(1)
        if outer_func not in KNOWN_FUNCTIONS:
            # Suggest the closest match
            lower_func = outer_func.lower()
            candidates = [f for f in KNOWN_FUNCTIONS if f.lower() == lower_func]
            suggestion = f" Did you mean '{candidates[0]}'?" if candidates else ''
            result.errors.append(
                f"Unknown function: '{outer_func}'.{suggestion} "
                'Check the function name — Power Automate function names are camelCase.'
            )

    return result


# Validate a set of correct and incorrect expressions
test_cases = [
    # Valid expressions
    "concat('Hello, ', 'world')",
    "formatDateTime(utcNow(), 'yyyy-MM-dd')",
    "if(equals(variables('Status'), 'Approved'), 'Yes', 'No')",
    "trim(toLower(triggerBody()?['FullName']))",
    # Invalid expressions — common beginner mistakes
    'concat("Hello, ", "world")',               # double quotes
    "concat('Hello'",                           # unbalanced parens
    "@equals(item()?['Status'], 'Open')",       # @ prefix
    "Concatt('a', 'b')",                        # wrong function name
    "toLower('hello')",                         # this one is actually valid
]

print('Expression Validation Results:')
print('=' * 70)
for expr in test_cases:
    v = validate_expression(expr)
    print(v)
    print()

## Exercises

Apply what you have learned. Each exercise gives a real Power Automate
scenario. Write the expression using the Python functions defined above
and verify using the evaluator.

In [ ]:
# Exercise 3.1: Safe greeting
#
# Scenario: An email trigger fires. The email sender's display name may or may
# not be present. Build a greeting:
#   - If display name exists: "Good morning, Alice!"
#   - If display name is empty/null: "Good morning!"
#
# Use: concat(), coalesce(), and string manipulation
# The evaluator context provides: display_name (may be empty string)

ev_31 = PowerAutomateEvaluator(context={
    'displayName': 'Alice Chen',
    'emptyName': '',
})

# YOUR CODE HERE
# Write two expressions:
#   expr_with_name: produce "Good morning, Alice Chen!"
#   expr_without_name: when displayName is empty, produce "Good morning!"

# Hint: coalesce(displayName, '') returns the first non-empty value
# Hint: if(empty(name), 'Good morning!', concat('Good morning, ', name, '!'))

expr_with_name = None     # replace None with your expression string
expr_without_name = None  # replace None with your expression string

exercise_3_1_with_name = expr_with_name
exercise_3_1_without_name = expr_without_name

In [ ]:
# AUTO-GRADED TEST — do not modify

def test_exercise_3_1():
    assert exercise_3_1_with_name is not None, \
        'Set expr_with_name to a Power Automate expression string'
    assert exercise_3_1_without_name is not None, \
        'Set expr_without_name to a Power Automate expression string'

    ev = PowerAutomateEvaluator(context={'displayName': 'Alice Chen', 'emptyName': ''})

    result_with = ev.evaluate(exercise_3_1_with_name)
    assert result_with == 'Good morning, Alice Chen!', \
        f'Expected "Good morning, Alice Chen!" but got {result_with!r}'

    result_without = ev.evaluate(exercise_3_1_without_name)
    assert result_without == 'Good morning!', \
        f'Expected "Good morning!" but got {result_without!r}'

    print('Exercise 3.1 passed.')
    print(f'  With name:    {result_with!r}')
    print(f'  Without name: {result_without!r}')

test_exercise_3_1()

In [ ]:
# Exercise 3.2: File name from date and status
#
# Scenario: A flow runs daily and generates a report file.
# The file name must be:  'report_20241115_approved.csv'
# Format: 'report_' + date in yyyyMMdd + '_' + status in lowercase + '.csv'
#
# Given:
#   - timestamp: '2024-11-15T09:32:00Z'
#   - status: 'Approved'
#
# Use: concat(), formatDateTime(), toLower()

ev_32 = PowerAutomateEvaluator(context={
    'timestamp': '2024-11-15T09:32:00Z',
    'status': 'Approved',
})

# YOUR CODE HERE
# Write an expression that produces: 'report_20241115_approved.csv'

expr_filename = None  # replace with your expression string

exercise_3_2 = expr_filename

In [ ]:
# AUTO-GRADED TEST — do not modify

def test_exercise_3_2():
    assert exercise_3_2 is not None, 'Set expr_filename to a Power Automate expression string'

    validation = validate_expression(exercise_3_2)
    assert validation.is_valid, \
        f'Expression has errors:\n' + '\n'.join(validation.errors)

    ev = PowerAutomateEvaluator(context={
        'timestamp': '2024-11-15T09:32:00Z',
        'status': 'Approved',
    })
    result = ev.evaluate(exercise_3_2)
    assert result == 'report_20241115_approved.csv', \
        f'Expected "report_20241115_approved.csv" but got {result!r}'

    print('Exercise 3.2 passed.')
    print(f'  Filename: {result!r}')

test_exercise_3_2()

In [ ]:
# Exercise 3.3: Validate your own expression
#
# Write a Power Automate expression that contains at least ONE of these
# common mistakes:
#   a) double quotes around a string literal
#   b) unbalanced parentheses
#   c) wrong function name
#
# Run it through validate_expression() and verify that an error is caught.
# Then fix the expression and verify it passes validation.

# YOUR CODE HERE
invalid_expression = None   # an expression with at least one error
fixed_expression = None     # the corrected version

exercise_3_3_invalid = invalid_expression
exercise_3_3_fixed = fixed_expression

In [ ]:
# AUTO-GRADED TEST — do not modify

def test_exercise_3_3():
    assert exercise_3_3_invalid is not None, 'Set invalid_expression to an expression string with at least one error'
    assert exercise_3_3_fixed is not None, 'Set fixed_expression to the corrected version'

    invalid_result = validate_expression(exercise_3_3_invalid)
    assert not invalid_result.is_valid, \
        f'Expected invalid_expression to fail validation but it passed: {exercise_3_3_invalid!r}'

    fixed_result = validate_expression(exercise_3_3_fixed)
    assert fixed_result.is_valid, \
        f'Expected fixed_expression to pass validation but got errors:\n' + '\n'.join(fixed_result.errors)

    print('Exercise 3.3 passed.')
    print(f'  Invalid expression errors: {invalid_result.errors}')
    print(f'  Fixed expression: {exercise_3_3_fixed!r} — VALID')

test_exercise_3_3()

## Summary

### Key Takeaways

1. **Power Automate expression functions map directly to Python built-ins.** The main differences are: function-call syntax for everything, single-quote string literals, and .NET datetime format codes.

2. **Five function categories cover almost all expression needs:** String, Date/Time, Logical, Type Conversion, and Collection.

3. **Type conversion is the most common fix for runtime errors.** When arithmetic or comparison fails, check whether inputs need `int()`, `float()`, or `string()` wrapping.

4. **Expressions evaluate inside-out.** The innermost function call produces a value that becomes the argument to the outer function. Trace from the inside to understand any expression.

5. **Three validation rules catch 80% of errors:** single quotes (not double), balanced parentheses, no `@` prefix in the Expression tab.

### What's Next

- **Exercise file:** `exercises/01_expression_builder.py` — practice writing expressions for real scenarios with automated validation
- **Guide 02:** Data Operations — Compose, Variables, Select, Filter array, Parse JSON
- **Module 04:** Branching and loops — where logical expressions drive Condition shapes and Apply to each controls

### Additional Resources

- [Power Automate expression function reference](https://learn.microsoft.com/en-us/azure/logic-apps/workflow-definition-language-functions-reference) — authoritative source for all functions
- [Workflow Definition Language format codes](https://learn.microsoft.com/en-us/dotnet/standard/base-types/custom-date-and-time-format-strings) — .NET datetime format string reference